# The XLM-R LR × step-budget sweep

XLM-R never trained on SIB-200 Yoruba. Three seeds at lr 2e-5 / 352 steps gave **0.057, 0.057,
0.105**, and 0.057 is exactly the macro-F1 of predicting the majority class on all 204 test items
(`reports/06-when-a-number-is-not-a-result.md`). All three sit below uniform random guessing and
below the untrained control at 0.107. So the open question is **not what XLM-R scores**. It is:

> **How often does it converge at all?**

A mean over collapsed seeds describes nothing — it averages a broken optimiser with a working one
and reports the result as a model property. So every cell here is reported as the **fraction of
seeds clearing the uniform-random bar**, with the per-seed scores printed next to it.

**Grid** — lr `{5e-6, 1e-5, 2e-5, 3e-5, 5e-5}` × steps `{352, 1056}` × 5 seeds = 10 cells,
50 fine-tuning runs. The LRs extend *downward*: 2e-5 already collapses, so the interesting
direction is gentler, not hotter. 352 and 1056 steps are 8 and 24 epochs at batch 16.

**Cost** — ~55 min on an A100, from the measured 32 s/seed at 352 steps. Check which card the
runtime actually handed out before trusting that: this account has been given both A100 and
Blackwell, and a T4 would be 5–10× worse.

**A negative across the whole grid is a finished result.** If nothing clears the bar, write it up
and stop — do not widen the grid hunting for a configuration that trains.

In [1]:
import os, sys
REPO = '/content/WashingtonCsed504'
FORK = 'https://github.com/patlkwok/WashingtonCsed504.git'   # YOUR fork, not upstream
if not os.path.exists(REPO):
    !git clone -q {FORK} {REPO}
sys.path.insert(0, f'{REPO}/src/a2-nlp')
import session; factory = session.start(prepare=False)   # no corpus: fine-tuning only

python 3.12.13 | NVIDIA A100-SXM4-80GB (85 GB, sm_80) | bf16: True
  installing seqeval ...
ready — cwd /content/WashingtonCsed504/src/a2-nlp


`session.start()` runs `git pull` against the fork, so a freshly pushed `ft_api.py` arrives here.
A live kernel still needs `importlib.reload(ft)` — `%autoreload` is the thing that dies on
Python 3.12.

In [2]:
import importlib, time
import numpy as np
import ft_api as ft
importlib.reload(ft)

GPU = ft.gpu_name()
print('ft_api', ft.API_VERSION, '| GPU', GPU)
if 'A100' not in GPU and 'PRO 6000' not in GPU:
    print('  NOTE: the ~55 min estimate assumes an A100. This is not one; expect it to differ.')

sib = ft.load_sib200('yor_Latn')

ft_api (1, 3) | GPU NVIDIA A100-SXM4-80GB


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


README.md:   0%|          | 0.00/47.9k [00:00<?, ?B/s]

train.tsv:   0%|          | 0.00/128k [00:00<?, ?B/s]

dev.tsv:   0%|          | 0.00/17.0k [00:00<?, ?B/s]

test.tsv:   0%|          | 0.00/37.6k [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

sib200/yor_Latn: train 701 validation 99 test 204 | 7 classes (chance 0.143) | 1.1% decomposed -> NFC


## The bar

Three different numbers get called "chance" on this task and they are not interchangeable, so
both of the ones that matter are **measured on the actual test set** rather than typed in as
constants. That is not pedantry: this project's recurring failure mode is a result produced by a
fixed arbitrary constant rather than by the thing under study, and it has six instances so far.

- **majority class** — what a model that answers one label for all 204 items scores. This is the
  collapse signature: the number XLM-R actually produced.
- **uniform random** — the bar this notebook counts against.
- **1/k = 0.143** — what `ft_api` uses for its own `degenerate` flag. It is a different bar from
  uniform random and usually the stricter one, so a cell can clear 0.133 and still be flagged.

All three are printed in order, next to the values quoted elsewhere in the project, so their
relationship is visible rather than assumed. Expect uniform random near 0.133 but do not be
alarmed by a couple of points either way — it depends on the class skew of the 204 test items,
not on a constant.

In [3]:
gold = sib['test']['label']
k = len(sib['labels'])

rng = np.random.default_rng(0)
UNIFORM = float(np.mean([ft.macro_f1(gold, rng.integers(0, k, len(gold)).tolist(), k)
                         for _ in range(400)]))
train = sib['train']['label']
MAJORITY = ft.macro_f1(gold, [max(set(train), key=train.count)] * len(gold), k)
CHANCE = 1 / k

print(f'{k} classes, {len(gold)} test items')
for name, val, note in sorted([('majority class', MAJORITY, 'project notes quote 0.057'),
                               ('uniform random', UNIFORM, 'project notes quote 0.133  <- the bar'),
                               ('1/k', CHANCE, 'ft_api degenerate flag')], key=lambda t: t[1]):
    print(f'  {name:<16}{val:.3f}   ({note})')

7 classes, 204 test items
  majority class  0.057   (project notes quote 0.057)
  uniform random  0.135   (project notes quote 0.133  <- the bar)
  1/k             0.143   (ft_api degenerate flag)


## The record this sweep overwrites

`record_tag` carries every setting that moves the number — but **not the seed count**. The
sweep's `lr=2e-5, steps=352` cell therefore has the *same tag* as the existing canonical XLM-R
row in the downstream table, and `reuse=True` will see 5 seeds where the file has 3, rerun, and
rewrite it.

That is the right outcome — 5 seeds on that cell is strictly better evidence than 3, and the
record carries its own `seeds` list so it stays self-describing. It is also recoverable either
way: `runs/*.json` is tracked, so the 3-seed version survives in git
(`git show HEAD:src/a2-nlp/runs/ft_sib200_yor_Latn_xlm-roberta-base_nfull_lr2e-05_st352_L128_nfc_ft.json`).

The cell below prints it first regardless, so the committed notebook holds the before-state.

In [4]:
for r in ft.results(task='sib200'):
    if r['model_slug'] == 'xlm-roberta-base':
        print(f"{r['tag']}\n  seeds {r['seeds']} -> {[round(s, 3) for s in r['scores']]}"
              f"\n  mean {r['mean']:.3f}  sd {r['sd']:.3f}  CI [{r['ci'][0]:.3f}, {r['ci'][1]:.3f}]"
              f"  degenerate={r['degenerate']}  gpu={r.get('gpu')}")

ft_sib200_yor_Latn_xlm-roberta-base_nfull_lr2e-05_st352_L128_nfc
  seeds [0, 1, 2] -> [0.057, 0.057, 0.105]
  mean 0.073  sd 0.022  CI [0.062, 0.084]  degenerate=True  gpu=NVIDIA A100-SXM4-80GB


## The sweep

`reuse=True` makes this notebook cheap to re-run: a completed cell with the same 5 seeds is read
back from disk instead of retrained. A Colab session that dies halfway resumes where it stopped,
**provided the records were saved off the disposable disk** — which is what the last cell is for.

In [5]:
XLMR  = 'FacebookAI/xlm-roberta-base'
LRS   = [5e-6, 1e-5, 2e-5, 3e-5, 5e-5]
STEPS = [352, 1056]
SEEDS = (0, 1, 2, 3, 4)
REUSE = True     # False to force every cell to retrain

t0 = time.time()
for steps in STEPS:
    for lr in LRS:
        ft.evaluate(XLMR, task='sib200', lr=lr, steps=steps, seeds=SEEDS,
                    data=sib, reuse=REUSE, label=f'XLM-R lr{lr:g} st{steps}')
print(f'\nsweep finished in {(time.time() - t0) / 60:.1f} min on {GPU}')

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.12GB            

...

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: FacebookAI/xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
classifier.dense.weight     | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.out_proj.weight  | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: FacebookAI/xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
classifier.dense.weight     | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.out_proj.weight  | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  XLM-R lr5e-05 st1056       macro_f1 0.057 +/-0.000 (seed sd) | 95% CI [0.046, 0.067] | n_train 701 | 95s/seed
    AT OR BELOW CHANCE (0.143) — this cell did not learn the task. Per-seed [0.057, 0.056, 0.057, 0.057, 0.057].
    Do not read it as a coverage result, and do not compare it with anything: a collapsed run
    moves further between reruns than any effect being tested.

sweep finished in 56.7 min on NVIDIA A100-SXM4-80GB


## The result

**Read the `converged` column, not the mean.** The mean is printed only so that a cell where some
seeds train and some do not is visibly not describable by one number.

In [6]:
rows = [r for r in ft.results(task='sib200')
        if r['model_slug'] == 'xlm-roberta-base' and len(r['seeds']) == len(SEEDS)]

print(f'{"lr":>8}{"steps":>7}{"converged":>11}{"mean":>8}{"best":>8}{"worst":>8}   per-seed')
print('-' * 78)
for r in sorted(rows, key=lambda r: (r['steps'], r['lr'])):
    s = r['scores']
    n = sum(x > UNIFORM for x in s)
    print(f'{r["lr"]:>8.0e}{r["steps"]:>7}{f"{n}/{len(s)}":>11}{r["mean"]:>8.3f}'
          f'{max(s):>8.3f}{min(s):>8.3f}   {[round(x, 3) for x in s]}')

hits = sum(sum(x > UNIFORM for x in r['scores']) for r in rows)
n_all = sum(len(r['scores']) for r in rows)
print(f'\n{hits}/{n_all} seeds across the whole grid cleared {UNIFORM:.3f}')
print(f'{sum(1 for r in rows if any(x > UNIFORM for x in r["scores"]))}/{len(rows)} '
      f'cells had at least one seed clear it')

      lr  steps  converged    mean    best   worst   per-seed
------------------------------------------------------------------------------
   5e-06    352        0/5   0.099   0.113   0.070   [0.07, 0.113, 0.104, 0.1, 0.108]
   1e-05    352        0/5   0.089   0.125   0.056   [0.125, 0.063, 0.097, 0.056, 0.103]
   2e-05    352        1/5   0.110   0.241   0.057   [0.057, 0.057, 0.105, 0.241, 0.089]
   3e-05    352        1/5   0.139   0.293   0.057   [0.125, 0.293, 0.057, 0.103, 0.118]
   5e-05    352        0/5   0.057   0.057   0.057   [0.057, 0.057, 0.057, 0.057, 0.057]
   5e-06   1056        5/5   0.295   0.372   0.251   [0.267, 0.372, 0.259, 0.325, 0.251]
   1e-05   1056        5/5   0.408   0.478   0.371   [0.407, 0.371, 0.392, 0.478, 0.391]
   2e-05   1056        4/5   0.394   0.499   0.119   [0.499, 0.444, 0.119, 0.425, 0.482]
   3e-05   1056        4/5   0.358   0.543   0.057   [0.543, 0.386, 0.409, 0.396, 0.057]
   5e-05   1056        0/5   0.057   0.057   0.056   [0.057, 

## What to do with it

**If nothing cleared the bar** — that is the finished result, and a better poster line than the
withdrawn 0.127. Write it up as *"XLM-R does not fine-tune on 701 Yoruba examples at any of 5
learning rates × 2 step budgets × 5 seeds"* and stop. Do not add LRs, do not add steps, do not
switch optimisers. The project's thesis does not depend on this: it is carried by seventeen
languages of pretraining and tokenizer measurement, and needs no working XLM-R fine-tune.

**If some cells converged** — report the fraction and the per-seed scores, never the mean over a
mixed cell. A converged XLM-R row also restores a real baseline for the downstream table, but the
comparison it feeds is still bounded: **never claim a difference below ~0.06 macro-F1 on 204 test
items.**

Either way the numbers belong in the poster via `ft.results()`, not by hand-copying from this
output.

In [7]:
session.save_results()   # asserts Drive is really mounted before copying

Mounted at /content/drive
copied 260 result files -> /content/drive/MyDrive/csed504-runs


260